# 00 Prepare Inputs

Prepare static input assets used by the AMR workflow.

Inputs/outputs live in `assets/`. This notebook validates or creates:

- `assets/counts_extended2.csv`: ranked pathogen priority list from Zenodo.
- `assets/antibiotic_synonym_lookup.csv`: antibiotic synonym lookup.
- `assets/antibiotic_coarsening_lookup.csv`: canonical antibiotic category lookup.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display


## Configuration


In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "assets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
ASSETS_DIR = PROJECT_ROOT / "assets"
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

PATHOGEN_PRIORITY_URL = "https://zenodo.org/records/18376711/files/counts_extended2.csv?download=1"
PATHOGEN_PRIORITY_OUTPUT = ASSETS_DIR / "counts_extended2.csv"
PATHOGEN_PRIORITY_REQUIRED_COLS = {"CanonicalSpecies", "pathogenic_index"}

SOURCE_XLSX = ASSETS_DIR / "antibiotic_list3.xlsx"
SOURCE_SHEET = "Sheet1"

ANTIBIOTIC_SYNONYM_OUTPUT = ASSETS_DIR / "antibiotic_synonym_lookup.csv"
ANTIBIOTIC_COARSENING_OUTPUT = ASSETS_DIR / "antibiotic_coarsening_lookup.csv"

CANONICAL_COL = "Antibiotic"
SYNONYM_COLS = ["alternative spelling", "Unnamed: 2", "Unnamed: 3", "Unnamed: 4"]
COARSENING_COLS = [
    "Antibiotic",
    "Class",
    "Simplified categories",
    "High level categories",
    "Origin",
]





## Prepare Pathogen Priority List

Download the ranked pathogen list if it is not already present in `assets/`, then validate the columns used by collection.


In [ ]:
if not PATHOGEN_PRIORITY_OUTPUT.exists():
    print(f"Downloading pathogen priority list to {PATHOGEN_PRIORITY_OUTPUT}")
    pathogen_priority_df = pd.read_csv(PATHOGEN_PRIORITY_URL)
    pathogen_priority_df.to_csv(PATHOGEN_PRIORITY_OUTPUT, index=False)
else:
    pathogen_priority_df = pd.read_csv(PATHOGEN_PRIORITY_OUTPUT)

missing_priority_cols = PATHOGEN_PRIORITY_REQUIRED_COLS - set(pathogen_priority_df.columns)
if missing_priority_cols:
    raise KeyError(
        f"{PATHOGEN_PRIORITY_OUTPUT} is missing required columns: {sorted(missing_priority_cols)}"
    )

pathogen_priority_df = pathogen_priority_df.copy()
pathogen_priority_df["CanonicalSpecies"] = pathogen_priority_df["CanonicalSpecies"].astype(str).str.strip()
pathogen_priority_df["pathogenic_index"] = pd.to_numeric(
    pathogen_priority_df["pathogenic_index"], errors="coerce"
)
pathogen_priority_df = pathogen_priority_df.dropna(subset=["CanonicalSpecies", "pathogenic_index"])
pathogen_priority_df = pathogen_priority_df.loc[pathogen_priority_df["CanonicalSpecies"] != ""].copy()
pathogen_priority_df = pathogen_priority_df.sort_values(
    ["pathogenic_index", "CanonicalSpecies"], ascending=[False, True]
).drop_duplicates(subset=["CanonicalSpecies"], keep="first")
pathogen_priority_df.to_csv(PATHOGEN_PRIORITY_OUTPUT, index=False)

print(f"Saved {len(pathogen_priority_df):,} pathogen priority rows to {PATHOGEN_PRIORITY_OUTPUT}")
display(pathogen_priority_df.head(10))


## Load and Validate Source Workbook


In [ ]:
source_df = pd.read_excel(SOURCE_XLSX, sheet_name=SOURCE_SHEET)
source_df.columns = [str(col).strip() for col in source_df.columns]

required_cols = {CANONICAL_COL, *SYNONYM_COLS, *COARSENING_COLS}
missing_cols = required_cols - set(source_df.columns)
if missing_cols:
    raise KeyError(f"{SOURCE_XLSX} is missing required columns: {sorted(missing_cols)}")

source_df = source_df.dropna(subset=[CANONICAL_COL]).copy()
source_df[CANONICAL_COL] = source_df[CANONICAL_COL].astype(str).str.strip()
source_df = source_df.loc[source_df[CANONICAL_COL] != ""].copy()

print(f"Loaded {len(source_df):,} antibiotic rows from {SOURCE_XLSX}")
display(source_df.head(10))


## Build Antibiotic Synonym Lookup

The output mirrors the bacteria synonym lookup shape: one synonym row per canonical antibiotic. The canonical antibiotic name is included as its own synonym so downstream query expansion can use a single lookup table.


In [ ]:
def clean_text(value):
    if pd.isna(value):
        return None
    text = " ".join(str(value).strip().split())
    return text or None


synonym_rows = []
for _, row in source_df.iterrows():
    canonical = clean_text(row[CANONICAL_COL])
    if canonical is None:
        continue
    variants = [canonical]
    variants.extend(clean_text(row[col]) for col in SYNONYM_COLS)
    for variant in variants:
        if variant is not None:
            synonym_rows.append(
                {
                    "antibiotic synonym": variant,
                    "antibiotic name": canonical,
                }
            )

antibiotic_synonym_df = (
    pd.DataFrame(synonym_rows)
    .drop_duplicates()
    .sort_values(["antibiotic name", "antibiotic synonym"])
    .reset_index(drop=True)
)

antibiotic_synonym_df.to_csv(ANTIBIOTIC_SYNONYM_OUTPUT, index=False)
print(f"Saved {len(antibiotic_synonym_df):,} synonym rows to {ANTIBIOTIC_SYNONYM_OUTPUT}")
display(antibiotic_synonym_df.head(20))


## Build Antibiotic Coarsening Lookup

This table stores the canonical antibiotic-to-category mapping used for grouped analyses and coarser visualizations.


In [ ]:
antibiotic_coarsening_df = source_df[COARSENING_COLS].copy()
for col in COARSENING_COLS:
    antibiotic_coarsening_df[col] = antibiotic_coarsening_df[col].map(clean_text)

antibiotic_coarsening_df = (
    antibiotic_coarsening_df
    .dropna(subset=["Antibiotic"])
    .drop_duplicates(subset=["Antibiotic"], keep="first")
    .sort_values("Antibiotic")
    .reset_index(drop=True)
)

antibiotic_coarsening_df.to_csv(ANTIBIOTIC_COARSENING_OUTPUT, index=False)
print(f"Saved {len(antibiotic_coarsening_df):,} coarsening rows to {ANTIBIOTIC_COARSENING_OUTPUT}")
display(antibiotic_coarsening_df.head(20))


## Quick Checks


In [ ]:
print("Canonical antibiotics:", antibiotic_coarsening_df["Antibiotic"].nunique())
print("Synonym variants:", antibiotic_synonym_df["antibiotic synonym"].nunique())
print("Classes:", antibiotic_coarsening_df["Class"].nunique(dropna=True))
print("Simplified categories:", antibiotic_coarsening_df["Simplified categories"].nunique(dropna=True))
print("High level categories:", antibiotic_coarsening_df["High level categories"].nunique(dropna=True))

duplicate_synonyms = antibiotic_synonym_df.groupby("antibiotic synonym")["antibiotic name"].nunique()
duplicate_synonyms = duplicate_synonyms[duplicate_synonyms > 1]
if duplicate_synonyms.empty:
    print("No synonym variants map to multiple canonical antibiotics.")
else:
    print("Synonym variants mapping to multiple canonical antibiotics:")
    display(duplicate_synonyms.reset_index(name="canonical_count"))
